# 1. Introduction

Ce notebook a pour objectif de nettoyer et préparer les données brutes collectées pour le Gold Futures et le Silver Futures.

Les étapes réalisées sont :

- Chargement des données brutes
- Vérification de la structure des données
- Nettoyage des colonnes
- Suppression des valeurs manquantes
- Suppression des doublons
- Sauvegarde des données nettoyées

# 2. Importation des bibliothèques

In [15]:
from pathlib import Path
import pandas as pd

# 3. Définition des chemins du projet

In [16]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier du projet :", PROJECT_ROOT)
print("Dossier données brutes :", RAW_DATA_DIR)
print("Dossier données traitées :", PROCESSED_DATA_DIR)

Dossier du projet : C:\FORMATION_CONTINUE_AND_TRAINING_SKILLS\420_A62_BB PROJET_DE_SYNTHESE\ai-trading-project
Dossier données brutes : C:\FORMATION_CONTINUE_AND_TRAINING_SKILLS\420_A62_BB PROJET_DE_SYNTHESE\ai-trading-project\data\raw
Dossier données traitées : C:\FORMATION_CONTINUE_AND_TRAINING_SKILLS\420_A62_BB PROJET_DE_SYNTHESE\ai-trading-project\data\processed


# 4. Chargement des données brutes

In [17]:
gold = pd.read_csv(RAW_DATA_DIR / "gold.csv")
silver = pd.read_csv(RAW_DATA_DIR / "silver.csv")

print("Gold :", gold.shape)
print("Silver :", silver.shape)

Gold : (11425, 6)
Silver : (11422, 6)


# 5. Aperçu des données

In [18]:
print("===== GOLD =====")
display(gold.head())

print("===== SILVER =====")
display(silver.head())

===== GOLD =====


,Price,Close,High,Low,Open,Volume
0,Ticker,GC=F,GC=F,GC=F,GC=F,GC=F
1,Datetime,NaN,NaN,NaN,NaN,NaN
2,2024-05-30 00:00:00-04:00,2333.5,2335.10009765625,2332.60009765625,2332.89990234375,0
3,2024-05-30 01:00:00-04:00,2323.699951171875,2333.699951171875,2320.800048828125,2333.0,617
4,2024-05-30 02:00:00-04:00,2331.699951171875,2331.699951171875,2322.800048828125,2323.39990234375,1913


===== SILVER =====


,Price,Close,High,Low,Open,Volume
0,Ticker,SI=F,SI=F,SI=F,SI=F,SI=F
1,Datetime,NaN,NaN,NaN,NaN,NaN
2,2024-05-30 00:00:00-04:00,31.815000534057617,31.915000915527344,31.770000457763672,31.780000686645508,0
3,2024-05-30 01:00:00-04:00,31.415000915527344,31.864999771118164,31.325000762939453,31.815000534057617,6755
4,2024-05-30 02:00:00-04:00,31.610000610351562,31.6299991607666,31.389999389648438,31.415000915527344,3980


## Structure des données

In [19]:
print("Colonnes Gold")
print(gold.columns)

print("\nTypes Gold")
print(gold.dtypes)

print("\nColonnes Silver")
print(silver.columns)

print("\nTypes Silver")
print(silver.dtypes)

Colonnes Gold
Index(['Price', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')

Types Gold
Price     str
Close     str
High      str
Low       str
Open      str
Volume    str
dtype: object

Colonnes Silver
Index(['Price', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='str')

Types Silver
Price     str
Close     str
High      str
Low       str
Open      str
Volume    str
dtype: object


# 6. Nettoyage des colonne

In [37]:
def clean_yfinance_columns(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    df = df.copy()

    # Si la première colonne n'est pas Datetime, on la renomme
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "Datetime"})

    # Garder seulement les lignes qui ressemblent à une vraie date
    df = df[df["Datetime"].astype(str).str.contains(r"\d{4}-\d{2}-\d{2}", regex=True, na=False)]

    # Garder seulement les 6 premières colonnes utiles
    df = df.iloc[:, :6]

    # Renommer proprement les colonnes
    df.columns = ["Datetime", "Close", "High", "Low", "Open", "Volume"]

    # Convertir les dates en forçant UTC
    df["Datetime"] = pd.to_datetime(df["Datetime"], errors="coerce", utc=True)

    # Convertir les colonnes numériques
    numeric_columns = ["Close", "High", "Low", "Open", "Volume"]

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Ajouter le symbole
    df["Ticker"] = ticker

    # Supprimer les lignes invalides
    df.dropna(inplace=True)

    return df

# 7. Application du nettoyage

In [38]:
gold = clean_yfinance_columns(gold, "GC=F")
silver = clean_yfinance_columns(silver, "SI=F")

print("Gold après nettoyage :", gold.shape)
print("Silver après nettoyage :", silver.shape)

display(gold.head())
display(silver.head())

Gold après nettoyage : (11423, 7)
Silver après nettoyage : (11420, 7)


,Datetime,Close,High,Low,Open,Volume,Ticker
0,2024-05-30 04:00:00+00:00,2333.500000,2335.100098,2332.600098,2332.899902,0,GC=F
1,2024-05-30 05:00:00+00:00,2323.699951,2333.699951,2320.800049,2333.000000,617,GC=F
2,2024-05-30 06:00:00+00:00,2331.699951,2331.699951,2322.800049,2323.399902,1913,GC=F
3,2024-05-30 07:00:00+00:00,2355.699951,2356.899902,2331.399902,2331.399902,45741,GC=F
4,2024-05-30 08:00:00+00:00,2356.300049,2357.300049,2330.600098,2355.800049,13770,GC=F


,Datetime,Close,High,Low,Open,Volume,Ticker
0,2024-05-30 04:00:00+00:00,31.815001,31.915001,31.770000,31.780001,0,SI=F
1,2024-05-30 05:00:00+00:00,31.415001,31.865000,31.325001,31.815001,6755,SI=F
2,2024-05-30 06:00:00+00:00,31.610001,31.629999,31.389999,31.415001,3980,SI=F
3,2024-05-30 07:00:00+00:00,31.650000,31.764999,31.584999,31.605000,3067,SI=F
4,2024-05-30 08:00:00+00:00,31.700001,31.740000,31.559999,31.665001,2580,SI=F


# 8. Vérification des types de données

In [39]:
print("===== TYPES GOLD =====")
print(gold.dtypes)

print("\n===== TYPES SILVER =====")
print(silver.dtypes)

===== TYPES GOLD =====
Datetime    datetime64[us, UTC]
Close                   float64
High                    float64
Low                     float64
Open                    float64
Volume                    int64
Ticker                      str
dtype: object

===== TYPES SILVER =====
Datetime    datetime64[us, UTC]
Close                   float64
High                    float64
Low                     float64
Open                    float64
Volume                    int64
Ticker                      str
dtype: object


# 9. Vérification des valeurs manquantes

In [40]:
print("Valeurs manquantes Gold")
print(gold.isnull().sum())

print("\nValeurs manquantes Silver")
print(silver.isnull().sum())

Valeurs manquantes Gold
Datetime    0
Close       0
High        0
Low         0
Open        0
Volume      0
Ticker      0
dtype: int64

Valeurs manquantes Silver
Datetime    0
Close       0
High        0
Low         0
Open        0
Volume      0
Ticker      0
dtype: int64


# 10. Suppression des valeurs manquantes

In [41]:
gold.dropna(inplace=True)
silver.dropna(inplace=True)

print("Gold après suppression NA :", gold.shape)
print("Silver après suppression NA :", silver.shape)

Gold après suppression NA : (11423, 7)
Silver après suppression NA : (11420, 7)


# 11. Vérification et suppression des doublons

In [42]:
print("Doublons Gold :", gold.duplicated().sum())
print("Doublons Silver :", silver.duplicated().sum())

gold.drop_duplicates(inplace=True)
silver.drop_duplicates(inplace=True)

print("Doublons supprimés")

Doublons Gold : 0
Doublons Silver : 0
Doublons supprimés


In [43]:
gold.to_csv(PROCESSED_DATA_DIR / "gold_clean.csv", index=False)
silver.to_csv(PROCESSED_DATA_DIR / "silver_clean.csv", index=False)

print("Prétraitement terminé")

Prétraitement terminé


# 12. Tri chronologique des données

In [44]:
gold.sort_values("Datetime", inplace=True)
silver.sort_values("Datetime", inplace=True)

gold.reset_index(drop=True, inplace=True)
silver.reset_index(drop=True, inplace=True)

display(gold.head())
display(silver.head())

,Datetime,Close,High,Low,Open,Volume,Ticker
0,2024-05-30 04:00:00+00:00,2333.500000,2335.100098,2332.600098,2332.899902,0,GC=F
1,2024-05-30 05:00:00+00:00,2323.699951,2333.699951,2320.800049,2333.000000,617,GC=F
2,2024-05-30 06:00:00+00:00,2331.699951,2331.699951,2322.800049,2323.399902,1913,GC=F
3,2024-05-30 07:00:00+00:00,2355.699951,2356.899902,2331.399902,2331.399902,45741,GC=F
4,2024-05-30 08:00:00+00:00,2356.300049,2357.300049,2330.600098,2355.800049,13770,GC=F


,Datetime,Close,High,Low,Open,Volume,Ticker
0,2024-05-30 04:00:00+00:00,31.815001,31.915001,31.770000,31.780001,0,SI=F
1,2024-05-30 05:00:00+00:00,31.415001,31.865000,31.325001,31.815001,6755,SI=F
2,2024-05-30 06:00:00+00:00,31.610001,31.629999,31.389999,31.415001,3980,SI=F
3,2024-05-30 07:00:00+00:00,31.650000,31.764999,31.584999,31.605000,3067,SI=F
4,2024-05-30 08:00:00+00:00,31.700001,31.740000,31.559999,31.665001,2580,SI=F


# 13. Sauvegarde des données nettoyées

In [45]:
gold.to_csv(PROCESSED_DATA_DIR / "gold_clean.csv", index=False)
silver.to_csv(PROCESSED_DATA_DIR / "silver_clean.csv", index=False)

print("Fichiers sauvegardés :")
print(PROCESSED_DATA_DIR / "gold_clean.csv")
print(PROCESSED_DATA_DIR / "silver_clean.csv")

Fichiers sauvegardés :
C:\FORMATION_CONTINUE_AND_TRAINING_SKILLS\420_A62_BB PROJET_DE_SYNTHESE\ai-trading-project\data\processed\gold_clean.csv
C:\FORMATION_CONTINUE_AND_TRAINING_SKILLS\420_A62_BB PROJET_DE_SYNTHESE\ai-trading-project\data\processed\silver_clean.csv


# 14. Vérification finale


In [46]:
gold_check = pd.read_csv(PROCESSED_DATA_DIR / "gold_clean.csv")
silver_check = pd.read_csv(PROCESSED_DATA_DIR / "silver_clean.csv")

print("Gold final :", gold_check.shape)
print("Silver final :", silver_check.shape)

display(gold_check.head())
display(silver_check.head())

Gold final : (11423, 7)
Silver final : (11420, 7)


,Datetime,Close,High,Low,Open,Volume,Ticker
0,2024-05-30 04:00:00+00:00,2333.500000,2335.100098,2332.600098,2332.899902,0,GC=F
1,2024-05-30 05:00:00+00:00,2323.699951,2333.699951,2320.800049,2333.000000,617,GC=F
2,2024-05-30 06:00:00+00:00,2331.699951,2331.699951,2322.800049,2323.399902,1913,GC=F
3,2024-05-30 07:00:00+00:00,2355.699951,2356.899902,2331.399902,2331.399902,45741,GC=F
4,2024-05-30 08:00:00+00:00,2356.300049,2357.300049,2330.600098,2355.800049,13770,GC=F


,Datetime,Close,High,Low,Open,Volume,Ticker
0,2024-05-30 04:00:00+00:00,31.815001,31.915001,31.770000,31.780001,0,SI=F
1,2024-05-30 05:00:00+00:00,31.415001,31.865000,31.325001,31.815001,6755,SI=F
2,2024-05-30 06:00:00+00:00,31.610001,31.629999,31.389999,31.415001,3980,SI=F
3,2024-05-30 07:00:00+00:00,31.650000,31.764999,31.584999,31.605000,3067,SI=F
4,2024-05-30 08:00:00+00:00,31.700001,31.740000,31.559999,31.665001,2580,SI=F


# 15. Conclusion

Le prétraitement des données a permis de :

- Charger les données brutes du Gold et du Silver
- Corriger la structure des colonnes issues de yfinance
- Convertir les dates et les colonnes numériques
- Supprimer les valeurs manquantes
- Supprimer les doublons
- Sauvegarder les données nettoyées dans le dossier `data/processed`

Les fichiers produits seront utilisés dans les prochaines étapes, notamment l'analyse exploratoire des données et la création des indicateurs techniques.